# P3: County Agriculture Merge

Primary merge per `MERGE.md` (§2, **P3**). Combines the five county-grain
agriculture tables into one wide table at **1 row per `county_fips` + `year`**.
Each source is **pivoted wide** first — its categorical descriptors
(commodity/statistic/animal/nutrient/source) become columns — so the final
`county_fips` + `year` join can't fan out.

**Inputs** (all `data/tabular/02_clean/agriculture/...`):
- `crop-yields-clean.csv` — pivoted by `commodity_detail` × `statistic` × `program`
- `livestock-inventory-clean.csv` — **`domain == "TOTAL"` layer only**, pivoted by `commodity_detail` × `statistic` × `program`
- `np-fertilizer-clean.csv` — pivoted by `nutrient` × `source` (kg)
- `np-manure-clean.csv` — pivoted by `animal_category` × `nutrient` (kg)
- `manure-animal-inventory-clean.csv` — pivoted by `animal` × `adjusted` (head)

**Output:** `data/03a_merge_primary/county-agriculture.csv`, one row per county + year.

**Design notes**
- **`program` is kept in the pivoted column name** for crop yields and livestock.
  In census years (2017/2022) both `CENSUS` and `SURVEY` report the same
  `commodity_detail`+`statistic`, which would otherwise collide; naming the two
  columns separately keeps both values instead of silently averaging them.
- **Only the livestock `TOTAL` layer is used.** The herd-size-band rows are the
  *same* operations re-counted into bands (`DATA.md`); including them would risk
  double-counting and explode the column count.
- **Roll-up rows with no `county_fips`** (`OTHER (COMBINED) COUNTIES`) are dropped —
  they have no county to join a station to.
- **Sparse by design.** Crop/livestock run annually 2015–2025; the N&P and
  manure-inventory tables run in ~5-year steps 1950–2017 (overlapping only at
  2017). The outer join is therefore sparse outside that window — expected, per
  `MERGE.md` §6.

In [ ]:
import os
import re

import pandas as pd
from functools import reduce

AG = "../../data/tabular/02_clean/agriculture"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/county-agriculture.csv"

KEY = ["county_fips", "year"]


def slug(x):
    """Lowercase, snake_case a categorical label for use in a column name."""
    s = str(x).strip().lower().replace("&", "and").replace("/", "_")
    s = re.sub(r"[(),.]", "", s)
    s = re.sub(r"\s+", "_", s)
    return s


def clean_county_year(df):
    """Standardize the join keys: 5-digit string county_fips (roll-ups dropped),
    integer year."""
    df = df.copy()
    df["county_fips"] = pd.to_numeric(df["county_fips"], errors="coerce")
    df = df.dropna(subset=["county_fips"])
    df["county_fips"] = df["county_fips"].astype(int).astype(str).str.zfill(5)
    df["year"] = df["year"].astype(int)
    return df


def pivot_wide(df, pivot_cols, value_col, prefix, suffix=""):
    """Pivot one long table to 1 row per county_fips+year. The (county, year,
    *pivot_cols) key is asserted unique first so pivot_table never silently
    aggregates. Column names are prefix__<slug>__<slug>...<suffix>."""
    df = clean_county_year(df)
    key = KEY + pivot_cols
    dup = df.duplicated(subset=key).sum()
    assert dup == 0, f"{prefix}: {dup} duplicate rows on {key} — pivot would aggregate"
    wide = df.pivot_table(index=KEY, columns=pivot_cols, values=value_col, aggfunc="mean")
    if isinstance(wide.columns, pd.MultiIndex):
        wide.columns = [prefix + "__" + "__".join(slug(p) for p in tup) + suffix for tup in wide.columns]
    else:
        wide.columns = [prefix + "__" + slug(c) + suffix for c in wide.columns]
    wide = wide.reset_index()
    print(f"{prefix}: {wide.shape[0]:,} county-years x {wide.shape[1] - 2} value cols")
    return wide

## Step 1: Crop yields

Pivoted by `commodity_detail` × `statistic` × `program`. Units vary by column
(BU / TONS / BU-per-ACRE) but are a deterministic function of
`commodity_detail`+`statistic`, so they're documented in `DATA.md` rather than
embedded in every column name.

In [ ]:
crop = pd.read_csv(f"{AG}/crop-yields-clean.csv")
crop_wide = pivot_wide(
    crop,
    pivot_cols=["commodity_detail", "statistic", "program"],
    value_col="value",
    prefix="cropyield",
)
crop_wide.head(3)

## Step 2: Livestock inventory (TOTAL layer)

Restricted to `domain == "TOTAL"`, then pivoted by
`commodity_detail` × `statistic` × `program`. `INVENTORY` values are head counts,
`OPERATIONS WITH INVENTORY` are farm counts.

In [ ]:
livestock = pd.read_csv(f"{AG}/livestock-inventory-clean.csv")
livestock_total = livestock[livestock["domain"] == "TOTAL"]
print(f"TOTAL layer: {len(livestock_total):,} of {len(livestock):,} rows")
livestock_wide = pivot_wide(
    livestock_total,
    pivot_cols=["commodity_detail", "statistic", "program"],
    value_col="value",
    prefix="livestock",
)
livestock_wide.head(3)

## Step 3: N & P from fertilizer

Pivoted by `nutrient` (N/P) × `source` (farm/nonfarm/total), kilograms.

In [ ]:
npfert = pd.read_csv(f"{AG}/np-fertilizer-clean.csv")
npfert_wide = pivot_wide(
    npfert,
    pivot_cols=["nutrient", "source"],
    value_col="value_kg",
    prefix="npfert",
    suffix="_kg",
)
npfert_wide.head(3)

## Step 4: N & P from manure

Pivoted by `animal_category` × `nutrient`, kilograms.

In [ ]:
npmanure = pd.read_csv(f"{AG}/np-manure-clean.csv")
npmanure_wide = pivot_wide(
    npmanure,
    pivot_cols=["animal_category", "nutrient"],
    value_col="value_kg",
    prefix="npmanure",
    suffix="_kg",
)
npmanure_wide.head(3)

## Step 5: Manure animal inventory

Pivoted by `animal` × `adjusted`, head counts. The `adjusted` flag
(`adj` = USDA slaughter-weight-adjusted, `raw` = unadjusted) is kept as part of
the column name so both series survive.

In [ ]:
manure_inv = pd.read_csv(f"{AG}/manure-animal-inventory-clean.csv")
manure_inv["adjusted"] = manure_inv["adjusted"].map({True: "adj", False: "raw"})
manure_inv_wide = pivot_wide(
    manure_inv,
    pivot_cols=["animal", "adjusted"],
    value_col="head_count",
    prefix="manureinv",
    suffix="_head",
)
manure_inv_wide.head(3)

## Step 6: Outer-merge on `county_fips` + `year`, then save

Each input is already 1 row per county-year, so an outer join on
`county_fips` + `year` unions the (county, year) coverage without any fan-out.

In [ ]:
frames = [crop_wide, livestock_wide, npfert_wide, npmanure_wide, manure_inv_wide]
df = reduce(lambda l, r: l.merge(r, on=KEY, how="outer"), frames)

assert not df.duplicated(subset=KEY).any(), "Output grain violated: duplicate (county_fips, year) rows"

df = df.sort_values(KEY).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"County-years: {len(df):,}  |  distinct counties: {df['county_fips'].nunique()}  |  "
      f"year range: {df['year'].min()}–{df['year'].max()}")

def block_coverage(prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "__")]
    return df[cols].notna().any(axis=1).sum()

print("\nCounty-years with any data, by source block:")
for pfx in ["cropyield", "livestock", "npfert", "npmanure", "manureinv"]:
    print(f"  {pfx:11s}: {block_coverage(pfx):>5,}")
overlap = df[(df['year'] >= 2015) & (df['year'] <= 2017)]
print(f"Rows in the 2015–2017 crop/N&P overlap window: {len(overlap):,}")

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")